In [10]:
import pandas as pd

deployments = pd.read_csv("../data/raw/deployments.csv")
meta_votes = deployments.groupby("platform")["votes_count"].sum()
print(meta_votes)
print("Total declared:", meta_votes.sum())

platform
aragon         26833
daohaus        51129
daostack       12331
realms       1067333
snapshot    64697595
tally        1440390
Name: votes_count, dtype: int64
Total declared: 67295611


In [11]:
platform_counts = []
for chunk in pd.read_csv("../data/raw/votes.csv", usecols=["platform"],
                         chunksize=1_000_000):
    platform_counts.append(chunk["platform"].value_counts())

rows_by_platform = pd.concat(platform_counts).groupby(level=0).sum()
coverage = pd.DataFrame({"declared": meta_votes, "rows": rows_by_platform}).fillna(0)
coverage.loc["total"] = coverage.sum()
coverage["coverage_pct"] = (100 * coverage["rows"] / coverage["declared"]).round(1)
print(coverage)

          declared      rows  coverage_pct
platform                                  
aragon       26833     26833         100.0
daohaus      51129     51129         100.0
daostack     12331     12331         100.0
realms     1067333   1067333         100.0
snapshot  64697595  64585436          99.8
tally      1440390   1597604         110.9
total     67295611  67340666         100.1


In [12]:
proposals_v4 = pd.Series({"snapshot": 244886, "tally": 37839, "realms": 21153,
                          "daohaus": 46936, "aragon": 15746, "daostack": 3573})
proposals_v5 = pd.read_csv("../data/raw/proposals.csv", usecols=["platform"])["platform"].value_counts()
print(pd.DataFrame({"v4": proposals_v4, "v5": proposals_v5,
                    "diff": proposals_v5 - proposals_v4}))

              v4      v5   diff
aragon     15746   15746      0
daohaus    46936   46936      0
daostack    3573    3573      0
realms     21153   21153      0
snapshot  244886  231845 -13041
tally      37839   32016  -5823


In [13]:
chunk_counts = []
for chunk in pd.read_csv("../data/raw/votes.csv",
                         usecols=["proposal_id", "platform"],
                         dtype={"proposal_id": str},
                         chunksize=1_000_000):
    snapshot_votes = chunk[chunk["platform"] == "snapshot"]
    chunk_counts.append(snapshot_votes["proposal_id"].value_counts())

actual_votes = pd.concat(chunk_counts).groupby(level=0).sum()
print("Snapshot proposals with votes:", len(actual_votes))
print("Max votes in a single proposal:", actual_votes.max())
print(actual_votes.value_counts().head(10))

Snapshot proposals with votes: 202954
Max votes in a single proposal: 2764428
count
1     84089
2     21711
3     10728
4      6807
5      5627
6      4167
7      3621
8      3063
10     2810
9      2745
Name: count, dtype: int64


In [14]:
snap = proposals[proposals["platform"] == "snapshot"]
dup = snap[snap["proposal_id"].duplicated(keep=False)]

print("Snapshot rows:", len(snap))
print("Distinct proposal_id:", snap["proposal_id"].nunique())
print("Rows with a repeated proposal_id:", len(dup))
print("Exact duplicate rows:", snap.duplicated().sum())
print("Repeated ids with different votes_count:",
      (dup.groupby("proposal_id")["votes_count"].nunique() > 1).sum())
print(dup.sort_values("proposal_id").head(6).to_string())

Snapshot rows: 231845
Distinct proposal_id: 231834
Rows with a repeated proposal_id: 22
Exact duplicate rows: 0
Repeated ids with different votes_count: 0
                                           proposal_id deployment_id  platform                 date  votes_count    total_vp                                      author platform_deployment_id
339159  QmPTrr1oqgXEfAvbwmTi5wG59gza1FLLHNUKJJZpajWGKC           NaN  snapshot  2021-05-01 11:08:47          2.0         NaN  0xaC3203D77823496E421aA7E88CDC2F6C387d6182  parallel-protocol.eth
97450   QmPTrr1oqgXEfAvbwmTi5wG59gza1FLLHNUKJJZpajWGKC      mimo.eth  snapshot  2021-05-01 11:08:47          2.0  11764304.0  0xac3203d77823496e421aa7e88cdc2f6c387d6182                    NaN
339160  QmS5qRKfAuAwX4fNoFRpyW4oW7xdSb5rUXyZcFW79W62Cw           NaN  snapshot  2021-04-28 11:28:14          6.0         NaN  0xaC3203D77823496E421aA7E88CDC2F6C387d6182  parallel-protocol.eth
97456   QmS5qRKfAuAwX4fNoFRpyW4oW7xdSb5rUXyZcFW79W62Cw      mimo.eth  snapsho

In [15]:
proposals = pd.read_csv("../data/raw/proposals.csv",
                        dtype={"proposal_id": str, "platform_deployment_id": str})
snapshot_declared = (proposals[proposals["platform"] == "snapshot"]
                     .drop_duplicates(subset="proposal_id")
                     .set_index("proposal_id")["votes_count"])

comparison = pd.DataFrame({"declared": snapshot_declared,
                           "actual": actual_votes}).fillna(0)
comparison["missing"] = comparison["declared"] - comparison["actual"]

capped = comparison["actual"] == 6000
empty = comparison["actual"] == 0

print("Proposals:", len(comparison))
print("Declared votes:", comparison["declared"].sum())
print("Actual votes:", comparison["actual"].sum())
print("Proposals capped at 6000:", capped.sum())
print("Missing votes in capped proposals:", comparison.loc[capped, "missing"].sum())
print("Proposals with 0 actual votes but declared > 0:",
      (empty & (comparison["declared"] > 0)).sum())
print("Missing votes in those proposals:", comparison.loc[empty, "missing"].sum())

Proposals: 231834
Declared votes: 64587411.0
Actual votes: 64585436.0
Proposals capped at 6000: 0
Missing votes in capped proposals: 0.0
Proposals with 0 actual votes but declared > 0: 55
Missing votes in those proposals: 70321.0


In [16]:
snapshot_props = proposals[proposals["platform"] == "snapshot"].copy()
snapshot_props["dao"] = snapshot_props["deployment_id"].fillna(
    snapshot_props["platform_deployment_id"])
snapshot_props["year"] = pd.to_datetime(snapshot_props["date"], errors="coerce").dt.year
snapshot_props["in_votes_csv"] = snapshot_props["proposal_id"].isin(actual_votes.index)

by_year = (snapshot_props.groupby(["year", "in_votes_csv"])["votes_count"]
           .sum().unstack(fill_value=0))
print(by_year)

missing_rows = snapshot_props[~snapshot_props["in_votes_csv"]
                              & (snapshot_props["votes_count"] > 0)]
print("DAOs affected:", missing_rows["dao"].nunique())
print(missing_rows.groupby("dao")["votes_count"].sum()
      .sort_values(ascending=False).head(10))

in_votes_csv    False       True 
year                             
2020              3.0     55358.0
2021          12444.0   1766802.0
2022          57324.0   5128160.0
2023              0.0  48451567.0
2024              0.0   8149852.0
2025            550.0    965413.0
DAOs affected: 19
dao
aavedao.eth                  32547.0
cakevote.eth                 23247.0
zora.eth                      7001.0
gyrodao.eth                   6945.0
networkfirstcommunity.eth      523.0
dforcenet.eth                   22.0
leviathannews.eth               13.0
tessnap.eth                      8.0
testsnapshotspace.eth            3.0
oaklanddao                       2.0
Name: votes_count, dtype: float64


In [17]:
snapshot_props["actual"] = snapshot_props["proposal_id"].map(actual_votes).fillna(0)

dao_coverage = snapshot_props.groupby("dao").agg(
    proposals=("proposal_id", "size"),
    proposals_with_votes=("in_votes_csv", "sum"),
    declared=("votes_count", "sum"),
    actual=("actual", "sum"),
)
dao_coverage["coverage"] = dao_coverage["actual"] / dao_coverage["declared"]

top_missing = missing_rows.groupby("dao")["votes_count"].sum().nlargest(10).index
print(dao_coverage.loc[top_missing])

                           proposals  proposals_with_votes   declared  \
dao                                                                     
aavedao.eth                      904                   900  3201862.0   
cakevote.eth                    3759                  3673   683173.0   
zora.eth                           3                     2    14368.0   
gyrodao.eth                        9                     8    17561.0   
networkfirstcommunity.eth         25                     0      523.0   
dforcenet.eth                     94                    90     1626.0   
leviathannews.eth                 10                     9      176.0   
tessnap.eth                        4                     0        8.0   
testsnapshotspace.eth              4                     1        4.0   
itokenpocket.eth                  20                    17     1079.0   

                              actual  coverage  
dao                                             
aavedao.eth              

In [18]:
present = comparison[comparison["actual"] > 0]
missing = comparison[(comparison["actual"] == 0) & (comparison["declared"] > 0)]

print("Max declared votes among present proposals:", present["declared"].max())
print("Present proposals with declared > 6000:", (present["declared"] > 6000).sum())
print("Missing proposals with declared > 6000:", (missing["declared"] > 6000).sum())
print(missing["declared"].describe())

large_missing = missing["declared"] > 6000
print("Missing votes, proposals > 6000:", missing.loc[large_missing, "declared"].sum())
print("Missing votes, proposals <= 6000:", missing.loc[~large_missing, "declared"].sum())

Max declared votes among present proposals: 2764428.0
Present proposals with declared > 6000: 1045
Missing proposals with declared > 6000: 8
count      55.000000
mean     1278.563636
std      2929.675571
min         1.000000
25%         2.000000
50%        15.000000
75%        25.000000
max      9056.000000
Name: declared, dtype: float64
Missing votes, proposals > 6000: 64269.0
Missing votes, proposals <= 6000: 6052.0


In [19]:
tally_parts = []
for chunk in pd.read_csv("../data/raw/votes.csv",
                         usecols=["vote_id", "platform"],
                         dtype={"vote_id": str},
                         chunksize=1_000_000):
    tally_parts.append(chunk.loc[chunk["platform"] == "tally", "vote_id"])

tally_ids = pd.concat(tally_parts)
print("Tally vote rows:", len(tally_ids))
print("Distinct vote_id:", tally_ids.nunique())

Tally vote rows: 1597604
Distinct vote_id: 1597604


In [20]:
tally_parts = []
for chunk in pd.read_csv("../data/raw/votes.csv",
                         usecols=["deployment_id", "proposal_id", "voter", "platform"],
                         dtype=str, chunksize=1_000_000):
    tally_parts.append(chunk[chunk["platform"] == "tally"])

tally_votes = pd.concat(tally_parts)
tally_votes["deployment_id"] = tally_votes["deployment_id"].str.lower()

print("Repeated (proposal_id, voter) pairs:",
      tally_votes.duplicated(subset=["proposal_id", "voter"]).sum())

tally_declared = (deployments[deployments["platform"] == "tally"]
                  .set_index("deployment_id")["votes_count"])
tally_actual = tally_votes.groupby("deployment_id").size()

tally_cmp = pd.DataFrame({"declared": tally_declared,
                          "actual": tally_actual}).fillna(0)
tally_cmp["extra"] = tally_cmp["actual"] - tally_cmp["declared"]
not_in_catalogue = ~tally_cmp.index.isin(tally_declared.index)

print("Votes in deployments missing from deployments.csv:",
      tally_cmp.loc[not_in_catalogue, "actual"].sum())
print("Deployments with more votes than declared:", (tally_cmp["extra"] > 0).sum())
print("Deployments with fewer votes than declared:", (tally_cmp["extra"] < 0).sum())
print(tally_cmp.sort_values("extra", ascending=False).head(5))

Repeated (proposal_id, voter) pairs: 559
Votes in deployments missing from deployments.csv: 0.0
Deployments with more votes than declared: 101
Deployments with fewer votes than declared: 934
                                                    declared    actual  \
deployment_id                                                            
eip155:10:0xcdf27f107725988f2261ce2256bdfcde8b3...    240303  351098.0   
eip155:42161:0xf07ded9dc292157749b6fd268e37df6e...    257389  283434.0   
eip155:324:0xb83ff6501214ddf40c91c9565d095400f3...      6258   23173.0   
eip155:7887:0x27926a991bb0193bf5b679bdb6cb3d3b6...     37426   50469.0   
eip155:324:0x76705327e682f2d96943280d99464ab612...     50216   54800.0   

                                                       extra  
deployment_id                                                 
eip155:10:0xcdf27f107725988f2261ce2256bdfcde8b3...  110795.0  
eip155:42161:0xf07ded9dc292157749b6fd268e37df6e...   26045.0  
eip155:324:0xb83ff6501214ddf40c91c9565

In [21]:
above = tally_cmp.loc[tally_cmp["extra"] > 0, "extra"].sum()
below = tally_cmp.loc[tally_cmp["extra"] < 0, "extra"].sum()
print("Votes above declared:", above)
print("Votes below declared:", below)

top_extra = tally_cmp.sort_values("extra", ascending=False).head(5).index
print(deployments.set_index("deployment_id").loc[top_extra, ["name", "unique_voters"]])

Votes above declared: 190512.0
Votes below declared: -33298.0
                                                                          name  \
deployment_id                                                                    
eip155:10:0xcdf27f107725988f2261ce2256bdfcde8b3...                        test   
eip155:42161:0xf07ded9dc292157749b6fd268e37df6e...               arbitrum core   
eip155:324:0xb83ff6501214ddf40c91c9565d095400f3...             zktokengovernor   
eip155:7887:0x27926a991bb0193bf5b679bdb6cb3d3b6...  kinto (proto governance) 2   
eip155:324:0x76705327e682f2d96943280d99464ab612...          zkprotocolgovernor   

                                                    unique_voters  
deployment_id                                                      
eip155:10:0xcdf27f107725988f2261ce2256bdfcde8b3...        78724.0  
eip155:42161:0xf07ded9dc292157749b6fd268e37df6e...       112713.0  
eip155:324:0xb83ff6501214ddf40c91c9565d095400f3...         3012.0  
eip155:7887:0x27926a991